# Coffee & Health — Análise Exploratória de Dados

**UC:** Análise de Dados e Big Data · Projeto A3 · 2026/2

| Nome | RA |
|---|---|
| | |
| | |

## 1. Introdução

Contexto, apresentação do dataset e objetivos do trabalho.

> Os dados são **sintéticos**: os resultados mostram associações, não causa e efeito nem evidência clínica.

## 2. Bibliotecas e configuração

| Biblioteca | Uso no projeto |
|---|---|
| `pandas` | Leitura do CSV e manipulação dos dados em DataFrames |
| `matplotlib` | Base dos gráficos |
| `seaborn` | Gráficos estatísticos sobre o matplotlib |
| `scikit-learn` | Separação treino/teste, Árvore de Decisão (Q8) e regressão linear (H1) |
| `scipy` | Testes estatísticos de correlação, ANOVA, qui-quadrado e teste *t* (Seção 8) |
| `python-dotenv` | Leitura dos valores fixos guardados no arquivo `.env` |
| `os` e `pathlib` | Variáveis de ambiente e caminhos de arquivo |

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

### Configuração

Os valores fixos do projeto ficam no arquivo `.env` (veja `.env.example`). O valor ao lado de cada variável só é usado quando o `.env` não existe.

In [ ]:
# Raiz do repositório: o notebook pode ser executado a partir de notebooks/ ou da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(RAIZ / ".env")

DATA_PATH = RAIZ / os.getenv("DATA_PATH", "data/synthetic_coffee_health_10000.csv")
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))
TOP_N_PAISES = int(os.getenv("TOP_N_PAISES", "5"))
FIG_LARGURA = float(os.getenv("FIG_LARGURA", "10"))
FIG_ALTURA = float(os.getenv("FIG_ALTURA", "5"))
CASAS_DECIMAIS = int(os.getenv("CASAS_DECIMAIS", "2"))
CASAS_CORRELACAO = int(os.getenv("CASAS_CORRELACAO", "4"))
MARGEM_TOPO_GRAFICO = float(os.getenv("MARGEM_TOPO_GRAFICO", "0.18"))
PADDING_ROTULO = int(os.getenv("PADDING_ROTULO", "4"))
BINS_HISTOGRAMA = int(os.getenv("BINS_HISTOGRAMA", "20"))
ALPHA_DISPERSAO = float(os.getenv("ALPHA_DISPERSAO", "0.25"))
TAMANHO_PONTO = int(os.getenv("TAMANHO_PONTO", "15"))
MAX_PROFUNDIDADE_ARVORE = int(os.getenv("MAX_PROFUNDIDADE_ARVORE", "3"))
FAIXAS_ETARIAS = [int(limite) for limite in os.getenv("FAIXAS_ETARIAS", "18,30,45,81").split(",")]
K_ANONIMATO = int(os.getenv("K_ANONIMATO", "5"))
LIMITES_CAFE = [float(v) for v in os.getenv("LIMITES_CAFE", "0,1,3,5,100").split(",")]
LIMITE_CORTE_CAFE = float(os.getenv("LIMITE_CORTE_CAFE", "3"))
LIMITE_CAFEINA_MG = float(os.getenv("LIMITE_CAFEINA_MG", "400"))
LIMITES_CAFEINA = [float(v) for v in os.getenv("LIMITES_CAFEINA", "0,100,200,300,400,500,1000").split(",")]
LIMITE_FC_ALTA = int(os.getenv("LIMITE_FC_ALTA", "85"))
CORTE_IDADE_RISCO = int(os.getenv("CORTE_IDADE_RISCO", "50"))
LIMITES_IMC = [float(v) for v in os.getenv("LIMITES_IMC", "0,18.5,25,30,100").split(",")]
FAIXA_IMC_TABELA = [int(v) for v in os.getenv("FAIXA_IMC_TABELA", "25,32").split(",")]
MINUTOS_POR_HORA = int(os.getenv("MINUTOS_POR_HORA", "60"))
LIMITE_P_EXIBICAO = float(os.getenv("LIMITE_P_EXIBICAO", "0.001"))

# Aparência padrão dos gráficos e das tabelas
sns.set_theme(style="whitegrid", rc={"figure.figsize": (FIG_LARGURA, FIG_ALTURA)})
pd.set_option("display.max_columns", None)

## 3. Carga dos dados

O CSV original nunca é alterado: `df_original` guarda os dados como foram lidos e `df` é a cópia usada nas transformações.

> Na leitura, o pandas converte o texto `None` da coluna `Health_Issues` em valor ausente (`NaN`). Esses registros são tratados na Q2.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"CSV não encontrado em {DATA_PATH}. Confira DATA_PATH no arquivo .env.")

df_original = pd.read_csv(DATA_PATH)
df = df_original.copy()

print(f"Arquivo carregado: {DATA_PATH.name}")
print(f"{df.shape[0]} registros e {df.shape[1]} colunas")

## 4. Q1 — Conhecendo o dataset

Quantos registros e colunas existem, quais são os tipos de dados, as cinco primeiras linhas e o resumo estatístico das colunas numéricas.

### Registros e colunas

In [ ]:
n_registros, n_colunas = df.shape

print(f"Registros: {n_registros}")
print(f"Colunas: {n_colunas}")

### Tipos de dados

`int64` são números inteiros, `float64` são números decimais e `str` (ou `object`, em versões mais antigas do pandas) são textos.

In [ ]:
df.dtypes.to_frame("Tipo no pandas")

In [ ]:
df.dtypes.astype(str).value_counts().to_frame("Quantidade de colunas")

### Cinco primeiras linhas

In [ ]:
df.head()

### Resumo estatístico das colunas numéricas

O `describe()` considera todas as colunas com tipo numérico. Três delas não são medidas:

- `ID` é só um identificador, então média e desvio padrão não têm significado;
- `Smoking` e `Alcohol_Consumption` são indicadores `0`/`1`, e a média equivale à proporção de registros com valor `1`.

In [ ]:
df.describe()

### Classificação das variáveis

O tipo no pandas indica como o dado está armazenado, não o que ele representa. A tabela abaixo classifica cada coluna pelo tipo de dado estatístico:

- **Numérica contínua:** pode assumir valores decimais dentro de um intervalo;
- **Numérica discreta:** valores inteiros, resultado de contagem ou registrados sem casas decimais;
- **Categórica nominal:** categorias sem ordem;
- **Categórica ordinal:** categorias com ordem (`Poor` < `Fair` < `Good` < `Excellent`; `Low` < `Medium` < `High`; `Mild` < `Moderate` < `Severe`);
- **Binária:** só dois valores, `0` e `1`;
- **Identificador:** distingue os registros e não entra nas análises.

In [ ]:
tipos_de_dado = {
    "ID": "Identificador",
    "Age": "Numérica discreta",
    "Gender": "Categórica nominal",
    "Country": "Categórica nominal",
    "Coffee_Intake": "Numérica contínua",
    "Caffeine_mg": "Numérica contínua",
    "Sleep_Hours": "Numérica contínua",
    "Sleep_Quality": "Categórica ordinal",
    "BMI": "Numérica contínua",
    "Heart_Rate": "Numérica discreta",
    "Stress_Level": "Categórica ordinal",
    "Physical_Activity_Hours": "Numérica contínua",
    "Health_Issues": "Categórica ordinal",
    "Occupation": "Categórica nominal",
    "Smoking": "Binária",
    "Alcohol_Consumption": "Binária",
}

classificacao = pd.DataFrame({
    "Tipo no pandas": df.dtypes.astype(str),
    "Valores distintos": df.nunique(),
    "Classificação": pd.Series(tipos_de_dado),
})
classificacao

In [ ]:
classificacao["Classificação"].value_counts().to_frame("Quantidade de colunas")

### Resposta

- O dataset tem **10.000 registros** e **16 colunas**; cada registro representa uma pessoa.
- No pandas, 5 colunas são inteiras (`int64`), 5 são decimais (`float64`) e 6 são de texto.
- Pela classificação estatística, são 5 numéricas contínuas, 2 numéricas discretas, 3 categóricas nominais, 3 categóricas ordinais, 2 binárias e 1 identificador.
- A idade média é de cerca de 35 anos, o consumo médio de café é de 2,5 e a média de sono é de 6,6 horas.
- A média de `Smoking` (0,20) e de `Alcohol_Consumption` (0,30) mostra que cerca de 20% dos registros são de fumantes e 30% de pessoas que consomem álcool.
- `Health_Issues` já aparece com valores ausentes (`NaN`) nas primeiras linhas; eles são analisados na Q2.

## 5. Q2 — Qualidade dos dados e ETL

O dataset tem valores ausentes ou registros duplicados? Em quais colunas, em que quantidade e como tratá-los sem modificar o arquivo original.

O diagnóstico é feito sobre `df_original`, que guarda os dados exatamente como foram lidos do CSV.

### Valores ausentes

Um valor ausente indica que a informação não foi registrada. No pandas, ele aparece como `NaN`.

- `isnull()` devolve `True` em cada célula vazia e `False` nas preenchidas.
- `.sum()` soma esses valores por coluna (`True` vale 1), resultando na quantidade de ausentes.
- `.mean()` calcula a proporção de ausentes em cada coluna; multiplicada por 100, vira percentual.

> **Atenção:** no CSV, os ausentes de `Health_Issues` estão escritos como o texto `None`. O `read_csv` converte esse texto em `NaN` automaticamente. Ausência **não** significa "sem problemas de saúde"; significa apenas que o dado não foi informado.

In [ ]:
ausentes = pd.DataFrame({
    "Valores ausentes": df_original.isnull().sum(),
    "% do total": df_original.isnull().mean() * 100,
})
ausentes

### Registros duplicados

Três verificações: linhas inteiras repetidas, linhas repetidas quando o `ID` é ignorado (a mesma pessoa cadastrada duas vezes com identificadores diferentes) e identificadores repetidos.

In [ ]:
linhas_duplicadas = df_original.duplicated().sum()
linhas_duplicadas_sem_id = df_original.drop(columns="ID").duplicated().sum()
ids_repetidos = df_original["ID"].duplicated().sum()

print(f"Linhas duplicadas: {linhas_duplicadas}")
print(f"Linhas duplicadas ignorando o ID: {linhas_duplicadas_sem_id}")
print(f"IDs repetidos: {ids_repetidos}")

### Consistência das categorias

Confere se as colunas de texto têm só as categorias previstas no dicionário de dados, sem variações de grafia.

In [ ]:
colunas_categoricas = ["Gender", "Country", "Sleep_Quality", "Stress_Level", "Health_Issues", "Occupation"]

for coluna in colunas_categoricas:
    categorias = sorted(df_original[coluna].dropna().unique())
    print(f"{coluna} ({len(categorias)}): {categorias}")

### Fluxo ETL

```
┌──────────────┐      ┌────────────────────────────┐      ┌──────────────────┐
│   EXTRAÇÃO   │      │       TRANSFORMAÇÃO        │      │      CARGA       │
│              │ ───► │                            │ ───► │                  │
│  CSV → df_   │      │  cópia → duplicados →      │      │  df: DataFrame   │
│  original    │      │  ausentes → ordinais →     │      │  limpo, usado    │
│              │      │  anonimização              │      │  nas análises    │
└──────────────┘      └────────────────────────────┘      └──────────────────┘
```

| Fase | Passo | O que é feito |
|---|---|---|
| Extração | Leitura | `pd.read_csv` lê o arquivo para `df_original` (seção 3). O CSV nunca é regravado. |
| Transformação | 1. Cópia | `df_original.copy()`: todas as mudanças são feitas na cópia. |
| Transformação | 2. Duplicados | `drop_duplicates()` remove linhas repetidas, se existirem. |
| Transformação | 3. Ausentes | Os vazios de `Health_Issues` recebem o rótulo `Não informado`. |
| Transformação | 4. Ordinais | `Sleep_Quality`, `Stress_Level` e `Health_Issues` passam a ter as categorias na ordem correta. |
| Transformação | 5. Anonimização | A coluna `ID` é removida, porque identifica o registro e não é usada nas análises. |
| Carga | DataFrame limpo | O resultado fica em `df`, usado em todas as seções seguintes. |

### Transformação

In [ ]:
ROTULO_NAO_INFORMADO = "Não informado"
ORDEM_SLEEP_QUALITY = ["Poor", "Fair", "Good", "Excellent"]
ORDEM_STRESS_LEVEL = ["Low", "Medium", "High"]
ORDEM_HEALTH_ISSUES = ["Mild", "Moderate", "Severe"]

# 1. Cópia: o original fica preservado em df_original
df = df_original.copy()

# 2. Duplicados
df = df.drop_duplicates()

# 3. Ausentes: vazio não significa "sem problema de saúde"
df["Health_Issues"] = df["Health_Issues"].fillna(ROTULO_NAO_INFORMADO)

# 4. Ordinais: categorias na ordem correta
df["Sleep_Quality"] = pd.Categorical(df["Sleep_Quality"], categories=ORDEM_SLEEP_QUALITY, ordered=True)
df["Stress_Level"] = pd.Categorical(df["Stress_Level"], categories=ORDEM_STRESS_LEVEL, ordered=True)
df["Health_Issues"] = pd.Categorical(df["Health_Issues"], categories=ORDEM_HEALTH_ISSUES + [ROTULO_NAO_INFORMADO])

# 5. Anonimização: remove o identificador
df = df.drop(columns="ID").reset_index(drop=True)

### Carga

Conferência do DataFrame limpo e de que o original continua intacto.

In [ ]:
print(f"Original: {df_original.shape[0]} registros e {df_original.shape[1]} colunas")
print(f"Limpo:    {df.shape[0]} registros e {df.shape[1]} colunas")
print(f"Valores ausentes no DataFrame limpo: {df.isnull().sum().sum()}")
print(f"Valores ausentes no original: {df_original.isnull().sum().sum()}")

In [ ]:
df["Health_Issues"].value_counts().to_frame("Registros")

In [ ]:
df.head()

### Resposta

- **Valores ausentes:** só a coluna `Health_Issues` tem valores ausentes: **5.941 registros (59,41%)**. As outras 15 colunas estão completas.
- **Duplicados:** não há registros duplicados, nem linhas inteiras, nem linhas iguais com `ID` diferente, nem `ID` repetido.
- **Categorias:** as colunas de texto têm só as categorias previstas, sem variações de grafia.

**Como tratar sem modificar o arquivo original:** o CSV é apenas lido, e todas as mudanças são feitas em `df`, uma cópia de `df_original`. Para os vazios de `Health_Issues` há duas opções, usadas conforme a análise:

1. **Rotular como `Não informado`** (feito acima): mantém os 10.000 registros nas análises que não dependem dessa coluna.
2. **Filtrar só os registros preenchidos**: é o que a Q10 pede, com `df[df["Health_Issues"] != ROTULO_NAO_INFORMADO]`.

Não é adequado preencher os vazios com a categoria mais frequente (`Mild`) nem tratá-los como "sem problema de saúde". No CSV esses campos trazem o texto `None`, mas o dicionário de dados orienta a não presumir que ausência significa ausência de problema, e alterar quase 60% da coluna distorceria qualquer comparação.

> **Para as próximas questões:** `Sleep_Quality`, `Stress_Level` e `Health_Issues` agora são colunas categóricas. Ao agrupar por elas, use `groupby(..., observed=True)` para que só apareçam as categorias e combinações que existem nos dados.

## 6. Análise Exploratória (EDA)

### Q3 — Perfil das pessoas

Qual é a distribuição dos registros por gênero (`Gender`) e qual é a média de idade (`Age`) em cada gênero?

#### Distribuição por gênero

In [ ]:
contagem_genero = df["Gender"].value_counts()
percentual_genero = df["Gender"].value_counts(normalize=True) * 100

distribuicao_genero = pd.DataFrame({
    "Registros": contagem_genero,
    "% do total": percentual_genero.round(CASAS_DECIMAIS),
})
distribuicao_genero

#### Idade por gênero

Além da média, a tabela traz a mediana, o desvio padrão e os extremos de idade em cada gênero. Comparar média e mediana ajuda a verificar se valores extremos puxam a média.

In [ ]:
ordem_generos = distribuicao_genero.index

idade_por_genero = (
    df.groupby("Gender")["Age"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínima",
        "max": "Máxima",
    })
    .reindex(ordem_generos)
    .round(CASAS_DECIMAIS)
)
idade_por_genero

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_genero = sns.color_palette(n_colors=len(distribuicao_genero))

# Gráfico 1: quantidade e percentual de registros por gênero
barras_qtd = ax1.bar(distribuicao_genero.index, distribuicao_genero["Registros"], color=cores_genero)
rotulos_qtd = [
    f"{int(linha['Registros'])}\n({linha['% do total']:.{CASAS_DECIMAIS}f}%)"
    for _, linha in distribuicao_genero.iterrows()
]
ax1.bar_label(barras_qtd, labels=rotulos_qtd, padding=PADDING_ROTULO)

# Gráfico 2: idade média por gênero com referência da média geral
barras_idade = ax2.bar(idade_por_genero.index, idade_por_genero["Média"], color=cores_genero)
ax2.bar_label(barras_idade, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)

media_geral_idade = df["Age"].mean()
ax2.axhline(
    media_geral_idade,
    color="gray",
    linestyle="--",
    label=f"Média geral ({media_geral_idade:.{CASAS_DECIMAIS}f} anos)",
)
ax2.legend(loc="upper right")

config_graficos = [
    (ax1, "Distribuição de registros por gênero", "Quantidade de registros", distribuicao_genero["Registros"].max()),
    (ax2, "Idade média por gênero", "Idade média (anos)", idade_por_genero["Média"].max()),
]
for ax, titulo, rotulo_y, valor_max in config_graficos:
    ax.set_title(titulo)
    ax.set_xlabel("Gênero")
    ax.set_ylabel(rotulo_y)
    ax.set_ylim(0, valor_max * (1 + MARGEM_TOPO_GRAFICO))
    ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Distribuição por gênero:** entre os 10.000 registros, `Female` concentra a maior parte (**5.001** registros; **50,01%**), seguido de perto por `Male` (**4.773**; **47,73%**), enquanto `Other` representa uma parcela menor (**226**; **2,26%**).
- **Média de idade por gênero:** as médias são muito próximas entre os três grupos — `Female` tem média de **35,03 anos**, `Male` de **34,89 anos** e `Other` de **34,50 anos** (diferença de cerca de 0,5 ano entre o maior e o menor grupo, próxima à média geral de **34,95 anos**).
- **Média vs. mediana:** em todos os gêneros, a mediana (`35` anos em `Female` e `34` anos em `Male` e `Other`) fica muito próxima da média, indicando distribuição etária equilibrada e sem distorção forte por valores extremos (idades de `18` a `80` anos em `Female` e `Male`, e de `18` a `64` em `Other`).

### Q4 — Consumo de café

Qual é o valor médio, mínimo e máximo de `Coffee_Intake`? Quais são os cinco países (`Country`) com maior média de consumo de café no dataset?

#### Estatísticas gerais de consumo

In [ ]:
resumo_cafe = pd.Series({
    "Média": df["Coffee_Intake"].mean(),
    "Mediana": df["Coffee_Intake"].median(),
    "Desvio padrão": df["Coffee_Intake"].std(),
    "Mínimo": df["Coffee_Intake"].min(),
    "Máximo": df["Coffee_Intake"].max(),
}).round(CASAS_DECIMAIS).to_frame("Coffee_Intake")

registros_zero = df["Coffee_Intake"].eq(0).sum()
print(f"Registros com consumo igual a zero: {registros_zero} ({registros_zero / len(df) * 100:.{CASAS_DECIMAIS}f}%)")
resumo_cafe

#### Países com maior média de consumo

In [ ]:
consumo_por_pais = (
    df.groupby("Country")["Coffee_Intake"]
    .agg(["count", "mean", "median", "std"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
    })
    .sort_values("Média", ascending=False)
    .round(CASAS_DECIMAIS)
)

top_paises = consumo_por_pais.head(TOP_N_PAISES)
print(f"Total de países: {len(consumo_por_pais)} | Menor média: {consumo_por_pais.index[-1]} ({consumo_por_pais['Média'].iloc[-1]:.{CASAS_DECIMAIS}f})")
top_paises

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: distribuição do consumo de café com média e mediana
ax1.hist(df["Coffee_Intake"], bins=BINS_HISTOGRAMA, edgecolor="white")
media_cafe = df["Coffee_Intake"].mean()
mediana_cafe = df["Coffee_Intake"].median()
ax1.axvline(media_cafe, color="crimson", linestyle="--", label=f"Média ({media_cafe:.{CASAS_DECIMAIS}f})")
ax1.axvline(mediana_cafe, color="darkorange", linestyle=":", label=f"Mediana ({mediana_cafe:.{CASAS_DECIMAIS}f})")
ax1.set_title("Distribuição de Coffee_Intake")
ax1.set_xlabel("Consumo de café (Coffee_Intake)")
ax1.set_ylabel("Quantidade de registros")
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: top países por consumo médio
top_ordenado = top_paises.iloc[::-1]
barras_paises = ax2.barh(top_ordenado.index, top_ordenado["Média"])
ax2.bar_label(barras_paises, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.set_title(f"Top {TOP_N_PAISES} países por média de consumo")
ax2.set_xlabel("Média de Coffee_Intake")
ax2.set_ylabel("País")
ax2.set_xlim(0, top_paises["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="y", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Resumo de `Coffee_Intake`:** média de **2,51**, mediana de **2,50**, valor mínimo de **0,0** (`558` registros, ou `5,58%`) e máximo de **8,2**. A proximidade entre média e mediana mostra distribuição equilibrada, sem forte distorção por valores extremos.
- **Cinco países com maior média:** **1º `Spain` (`2,60`)**, **2º `Norway` (`2,59`)**, **3º `UK` (`2,58`)**, **4º `China` (`2,57`)** e **5º `USA` (`2,56`)**.
- **Amplitude entre países:** as médias dos 20 países variam em um intervalo curto, de `2,41` (`Belgium`) a `2,60` (`Spain`), indicando consumo médio semelhante entre os países neste dataset sintético.

### Q5 — Café e cafeína

Existe relação entre `Coffee_Intake` e `Caffeine_mg`? Calcule a correlação e represente as variáveis em um gráfico de dispersão. Explique o resultado.

#### Correlação de Pearson

In [ ]:
matriz_corr = df[["Coffee_Intake", "Caffeine_mg"]].corr().round(CASAS_CORRELACAO)
correlacao_cafe_cafeina = df["Coffee_Intake"].corr(df["Caffeine_mg"])

consumo_positivo = df[df["Coffee_Intake"] > 0]
mg_por_unidade = (consumo_positivo["Caffeine_mg"] / consumo_positivo["Coffee_Intake"]).mean()

print(f"Correlação de Pearson: {correlacao_cafe_cafeina:.{CASAS_CORRELACAO}f}")
print(f"Razão média (entre registros com consumo > 0): {mg_por_unidade:.{CASAS_DECIMAIS}f} mg por unidade de Coffee_Intake")
matriz_corr

#### Gráfico de dispersão

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_LARGURA, FIG_ALTURA))

sns.regplot(
    data=df,
    x="Coffee_Intake",
    y="Caffeine_mg",
    scatter_kws={"alpha": ALPHA_DISPERSAO, "s": TAMANHO_PONTO},
    line_kws={"color": "crimson"},
    ax=ax,
)
ax.set_title(f"Relação entre consumo de café e cafeína (r = {correlacao_cafe_cafeina:.{CASAS_CORRELACAO}f})")
ax.set_xlabel("Consumo de café (Coffee_Intake)")
ax.set_ylabel("Cafeína (mg)")

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Correlação encontrada:** a correlação de Pearson entre `Coffee_Intake` e `Caffeine_mg` é **r = 0,9998**, uma relação linear positiva praticamente perfeita.
- **Explicação do resultado:** os pontos do gráfico de dispersão acompanham uma linha reta crescente com dispersão mínima em torno da tendência. Cada unidade de `Coffee_Intake` corresponde, em média, a cerca de **95 mg de cafeína** (valor próximo ao de uma xícara padrão), indicando que `Caffeine_mg` foi gerada de forma diretamente proporcional ao consumo de café no dataset sintético e traz praticamente a mesma informação.

### Q6 — Consumo de café e sono

Qual é a média de `Coffee_Intake` para cada categoria de `Sleep_Quality`? Os registros classificados com sono `Poor` apresentam consumo médio maior ou menor do que aqueles com sono `Excellent`?

#### Consumo por qualidade do sono

In [ ]:
cafe_por_sono = (
    df.groupby("Sleep_Quality", observed=True)["Coffee_Intake"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínimo",
        "max": "Máximo",
    })
    .round(CASAS_DECIMAIS)
)

dif_poor_excellent = cafe_por_sono.loc["Poor", "Média"] - cafe_por_sono.loc["Excellent", "Média"]
print(f"Diferença média (Poor - Excellent): {dif_poor_excellent:.{CASAS_DECIMAIS}f}")
cafe_por_sono

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_sono = sns.color_palette(n_colors=len(cafe_por_sono))

# Gráfico 1: média de Coffee_Intake por Sleep_Quality
barras_sono = ax1.bar(cafe_por_sono.index.astype(str), cafe_por_sono["Média"], color=cores_sono)
ax1.bar_label(barras_sono, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.axhline(
    df["Coffee_Intake"].mean(),
    color="gray",
    linestyle="--",
    label=f"Média geral ({df['Coffee_Intake'].mean():.{CASAS_DECIMAIS}f})",
)
ax1.set_title("Consumo médio de café por qualidade do sono")
ax1.set_xlabel("Qualidade do sono")
ax1.set_ylabel("Média de Coffee_Intake")
ax1.set_ylim(0, cafe_por_sono["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: distribuição (boxplot) para comparar medianas e dispersão
sns.boxplot(data=df, x="Sleep_Quality", y="Coffee_Intake", hue="Sleep_Quality", dodge=False, legend=False, ax=ax2)
ax2.set_title("Distribuição do consumo por qualidade do sono")
ax2.set_xlabel("Qualidade do sono")
ax2.set_ylabel("Coffee_Intake")
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Média de `Coffee_Intake` por categoria:**
  - `Poor` (`961` registros): média **2,98** (mediana **3,00**)
  - `Fair` (`2.050` registros): média **2,76** (mediana **2,80**)
  - `Good` (`5.637` registros): média **2,45** (mediana **2,40**)
  - `Excellent` (`1.352` registros): média **2,05** (mediana **2,00**)
- **Comparação `Poor` vs. `Excellent`:** os registros com sono **`Poor`** apresentam consumo médio **maior** do que aqueles com sono **`Excellent`** (`2,98` contra `2,05`, uma diferença de **0,93** unidade).
- **Padrão e outliers:** a média cai de forma contínua conforme a qualidade do sono melhora, e a mediana acompanha a média de perto em todas as faixas, mostrando que a diferença reflete o comportamento geral dos grupos e não apenas valores extremos.

### Q7 — Sono e estresse

Qual é a média de horas de sono (`Sleep_Hours`) em cada nível de estresse (`Stress_Level`)? Apresente um gráfico comparativo e descreva o padrão encontrado.

#### Horas de sono por nível de estresse

In [ ]:
sono_por_estresse = (
    df.groupby("Stress_Level", observed=True)["Sleep_Hours"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .rename(columns={
        "count": "Registros",
        "mean": "Média",
        "median": "Mediana",
        "std": "Desvio padrão",
        "min": "Mínimo",
        "max": "Máximo",
    })
    .round(CASAS_DECIMAIS)
)
sono_por_estresse

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_estresse = sns.color_palette(n_colors=len(sono_por_estresse))

# Gráfico 1: média de horas de sono por nível de estresse
barras_estresse = ax1.bar(sono_por_estresse.index.astype(str), sono_por_estresse["Média"], color=cores_estresse)
ax1.bar_label(barras_estresse, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.axhline(
    df["Sleep_Hours"].mean(),
    color="gray",
    linestyle="--",
    label=f"Média geral ({df['Sleep_Hours'].mean():.{CASAS_DECIMAIS}f} h)",
)
ax1.set_title("Média de horas de sono por nível de estresse")
ax1.set_xlabel("Nível de estresse")
ax1.set_ylabel("Média de Sleep_Hours (h)")
ax1.set_ylim(0, sono_por_estresse["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

# Gráfico 2: distribuição das horas de sono por nível de estresse
sns.boxplot(data=df, x="Stress_Level", y="Sleep_Hours", hue="Stress_Level", dodge=False, legend=False, ax=ax2)
ax2.set_title("Distribuição de horas de sono por estresse")
ax2.set_xlabel("Nível de estresse")
ax2.set_ylabel("Sleep_Hours (h)")
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Média de `Sleep_Hours` por `Stress_Level`:**
  - `Low` (`6.989` registros): média **7,25 h** (mediana **7,10 h**; de `6,0` a `10,0 h`)
  - `Medium` (`2.050` registros): média **5,58 h** (mediana **5,60 h**; de `5,0` a `6,0 h`)
  - `High` (`961` registros): média **4,45 h** (mediana **4,60 h**; de `3,0` a `5,0 h`)
- **Padrão encontrado:** há uma associação inversa clara e acentuada — conforme o nível de estresse aumenta de `Low` para `High`, a média de sono cai **2,80 horas**. Como mostram os boxplots e os limites mínimo e máximo, os três níveis ocupam faixas praticamente separadas de horas de sono no dataset sintético, com média e mediana muito próximas em todos os grupos.

### Q8 — Classificação com Árvore de Decisão

Modelo simples com `DecisionTreeClassifier` para prever `Sleep_Quality` a partir de `Coffee_Intake`, `Caffeine_mg`, `Age`, `BMI`, `Heart_Rate` e `Physical_Activity_Hours`, sem utilizar `ID`.

#### Separação em treino e teste

In [ ]:
ATRIBUTOS_Q8 = [
    "Coffee_Intake",
    "Caffeine_mg",
    "Age",
    "BMI",
    "Heart_Rate",
    "Physical_Activity_Hours",
]
ALVO_Q8 = "Sleep_Quality"

# Confirma que o identificador não entra nos preditores
assert "ID" not in ATRIBUTOS_Q8 and "ID" not in df.columns

X = df[ATRIBUTOS_Q8]
y = df[ALVO_Q8]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

classe_majoritaria = y_train.mode()[0]
acuracia_baseline = (y_test == classe_majoritaria).mean()

print(f"Treino: {len(X_train)} registros | Teste: {len(X_test)} registros")
print(f"Baseline (sempre prever '{classe_majoritaria}'): {acuracia_baseline * 100:.{CASAS_DECIMAIS}f}%")

#### Treinamento e comparação de desempenho

Além da árvore padrão (`max_depth=None`), testamos uma árvore com profundidade limitada (`MAX_PROFUNDIDADE_ARVORE`) para observar o efeito do ajuste excessivo (*overfitting*) aos dados de treino.

In [ ]:
modelo_livre = DecisionTreeClassifier(random_state=RANDOM_STATE)
modelo_livre.fit(X_train, y_train)

modelo_podado = DecisionTreeClassifier(max_depth=MAX_PROFUNDIDADE_ARVORE, random_state=RANDOM_STATE)
modelo_podado.fit(X_train, y_train)

comparacao_modelos = pd.DataFrame([
    {
        "Modelo": f"Baseline (classe majoritária: {classe_majoritaria})",
        "Acurácia treino (%)": (y_train == classe_majoritaria).mean() * 100,
        "Acurácia teste (%)": acuracia_baseline * 100,
    },
    {
        "Modelo": "Árvore de Decisão (max_depth=None)",
        "Acurácia treino (%)": accuracy_score(y_train, modelo_livre.predict(X_train)) * 100,
        "Acurácia teste (%)": accuracy_score(y_test, modelo_livre.predict(X_test)) * 100,
    },
    {
        "Modelo": f"Árvore de Decisão (max_depth={MAX_PROFUNDIDADE_ARVORE})",
        "Acurácia treino (%)": accuracy_score(y_train, modelo_podado.predict(X_train)) * 100,
        "Acurácia teste (%)": accuracy_score(y_test, modelo_podado.predict(X_test)) * 100,
    },
]).round(CASAS_DECIMAIS)

comparacao_modelos

In [ ]:
pred_podada = modelo_podado.predict(X_test)
print(classification_report(y_test, pred_podada, labels=ORDEM_SLEEP_QUALITY, zero_division=0))

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: matriz de confusão do modelo podado no conjunto de teste
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_podada,
    labels=ORDEM_SLEEP_QUALITY,
    ax=ax1,
    colorbar=False,
    cmap="Blues",
)
ax1.set_title(f"Matriz de confusão (max_depth={MAX_PROFUNDIDADE_ARVORE})")
ax1.set_xlabel("Previsto")
ax1.set_ylabel("Real")
ax1.grid(False)

# Gráfico 2: importância dos atributos no modelo podado
importancias = pd.Series(modelo_podado.feature_importances_, index=ATRIBUTOS_Q8).sort_values() * 100
barras_imp = ax2.barh(importancias.index, importancias.values)
ax2.bar_label(barras_imp, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax2.set_title(f"Importância dos atributos (max_depth={MAX_PROFUNDIDADE_ARVORE})")
ax2.set_xlabel("Importância (%)")
ax2.set_xlim(0, importancias.max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="y", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Acurácia obtida:** com divisão estratificada (`80%` treino / `20%` teste, `RANDOM_STATE=42`), a Árvore de Decisão padrão (`max_depth=None`) atingiu **100,00%** no treino e **38,90%** no teste, evidenciando *overfitting*. Ao limitar a profundidade (`max_depth=3`), a acurácia no teste sobe para **56,40%** (`56,51%` no treino).
- **Interpretação:** a acurácia de **56,40%** apenas empata com a *baseline* majoritária (prever sempre `Good`, que representa `56,37%` dos dados). Como mostra a matriz de confusão, mesmo usando `Caffeine_mg` (`88,38%` da importância na árvore podada), o modelo quase sempre prevê `Good`.
- **Por que isso ocorre:** nos dados sintéticos, `Sleep_Quality` está ligada principalmente a `Sleep_Hours` (que não entra entre os seis preditores da Q8), enquanto `Coffee_Intake`, `Caffeine_mg`, `Age`, `BMI`, `Heart_Rate` e `Physical_Activity_Hours` se sobrepõem fortemente entre as quatro classes.

### Q9 — Análise conjunta de consumo, sono e estresse

Agrupamento por `Stress_Level` e `Sleep_Quality` com quantidade de pessoas, média de `Coffee_Intake` e média de `Sleep_Hours`.

#### Agrupamento por estresse e qualidade do sono

In [ ]:
analise_conjunta = (
    df.groupby(["Stress_Level", "Sleep_Quality"], observed=True)
    .agg(
        Pessoas=("Coffee_Intake", "count"),
        **{
            "Café (média)": ("Coffee_Intake", "mean"),
            "Café (mediana)": ("Coffee_Intake", "median"),
            "Sono h (média)": ("Sleep_Hours", "mean"),
            "Sono h (mediana)": ("Sleep_Hours", "median"),
        },
    )
    .assign(**{"% do total": lambda t: t["Pessoas"] / len(df) * 100})
    .sort_values("Café (média)", ascending=False)
    .round(CASAS_DECIMAIS)
)

total_combinacoes = len(ORDEM_STRESS_LEVEL) * len(ORDEM_SLEEP_QUALITY)
print(f"Combinações existentes nos dados: {len(analise_conjunta)} de {total_combinacoes} possíveis")
analise_conjunta

#### Visualização

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))

# Gráfico 1: mapa de calor com média de café e n em cada combinação possível
tabela_calor = df.pivot_table(index="Stress_Level", columns="Sleep_Quality", values="Coffee_Intake", aggfunc="mean", observed=False)
contagem_calor = df.pivot_table(index="Stress_Level", columns="Sleep_Quality", values="Coffee_Intake", aggfunc="count", observed=False)
anotacoes = tabela_calor.copy().astype(object)
for linha in tabela_calor.index:
    for coluna in tabela_calor.columns:
        valor = tabela_calor.loc[linha, coluna]
        qtd = int(contagem_calor.loc[linha, coluna])
        anotacoes.loc[linha, coluna] = f"{valor:.{CASAS_DECIMAIS}f}\n(n={qtd})" if pd.notna(valor) else "—"

sns.heatmap(tabela_calor, annot=anotacoes, fmt="", cmap="YlOrBr", cbar=False, linewidths=1, linecolor="white", ax=ax1)
ax1.set_title("Média de café e contagem (Estresse × Sono)")
ax1.set_xlabel("Qualidade do sono")
ax1.set_ylabel("Nível de estresse")

# Gráfico 2: comparação direta dos 4 grupos existentes
rotulos_grupos = [f"{est}\n{sono}" for est, sono in analise_conjunta.index]
cores_q9 = sns.color_palette(n_colors=len(analise_conjunta))
barras_q9 = ax2.bar(rotulos_grupos, analise_conjunta["Café (média)"], color=cores_q9)
rotulos_q9 = [
    f"{linha['Café (média)']:.{CASAS_DECIMAIS}f}\n(n={int(linha['Pessoas'])})"
    for _, linha in analise_conjunta.iterrows()
]
ax2.bar_label(barras_q9, labels=rotulos_q9, padding=PADDING_ROTULO)
ax2.set_title("Consumo médio de café nos grupos existentes")
ax2.set_xlabel("Nível de estresse / Qualidade do sono")
ax2.set_ylabel("Média de Coffee_Intake")
ax2.set_ylim(0, analise_conjunta["Café (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Grupos existentes:** das 12 combinações possíveis entre `Stress_Level` e `Sleep_Quality`, apenas **4** aparecem nos dados:
  1. `High` × `Poor`: **961** pessoas (`9,61%`) | café médio **2,98** | sono médio **4,45 h**
  2. `Medium` × `Fair`: **2.050** pessoas (`20,50%`) | café médio **2,76** | sono médio **5,58 h**
  3. `Low` × `Good`: **5.637** pessoas (`56,37%`) | café médio **2,45** | sono médio **6,92 h**
  4. `Low` × `Excellent`: **1.352** pessoas (`13,52%`) | café médio **2,05** | sono médio **8,59 h**
- **Maior consumo médio de café:** ocorre na combinação **`High` × `Poor`** (`2,98`), seguida por **`Medium` × `Fair`** (`2,76`).
- **Interpretação com base no tamanho dos grupos:** o grupo com maior consumo médio (`High × Poor`) é o **menor** dos quatro grupos existentes (menos de `10%` da amostra), enquanto mais da metade das pessoas (`56,37%`) concentra-se em `Low × Good`. As outras 8 combinações têm contagem zero porque, na geração sintética da base, `Stress_Level` e `Sleep_Quality` seguem as mesmas faixas de `Sleep_Hours` (`High` ↔ `Poor`, `Medium` ↔ `Fair`, `Low` ↔ `Good`/`Excellent`).

### Q10 — Possíveis associações com problemas de saúde

Considerando **somente os registros em que `Health_Issues` está preenchido** (`Mild`, `Moderate` e `Severe`), comparar quantidade de registros, médias de `Coffee_Intake`, `Caffeine_mg`, `BMI` e `Heart_Rate`, e a distribuição percentual de `Smoking`.

#### Filtro dos registros preenchidos e resumo por gravidade

In [ ]:
df_saude = df[df["Health_Issues"] != ROTULO_NAO_INFORMADO].copy()

print(f"Registros com Health_Issues preenchido: {len(df_saude)} de {len(df)} ({len(df_saude) / len(df) * 100:.{CASAS_DECIMAIS}f}%)")

resumo_saude = (
    df_saude.groupby("Health_Issues", observed=True)
    .agg(
        Registros=("Coffee_Intake", "count"),
        **{
            "Café (média)": ("Coffee_Intake", "mean"),
            "Café (mediana)": ("Coffee_Intake", "median"),
            "Cafeína mg (média)": ("Caffeine_mg", "mean"),
            "BMI (média)": ("BMI", "mean"),
            "Heart_Rate (média)": ("Heart_Rate", "mean"),
            "Fumantes (%)": ("Smoking", lambda s: s.mean() * 100),
        },
    )
    .assign(**{"% preenchidos": lambda t: t["Registros"] / len(df_saude) * 100})
    .round(CASAS_DECIMAIS)
)
resumo_saude

#### Distribuição percentual de tabagismo (`Smoking`)

In [ ]:
dist_smoking = (
    pd.crosstab(df_saude["Health_Issues"], df_saude["Smoking"], normalize="index") * 100
).rename(columns={0: "Não fumante (0) %", 1: "Fumante (1) %"}).round(CASAS_DECIMAIS)

dist_smoking

#### Visualização

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(FIG_LARGURA, FIG_ALTURA))
categorias_saude = resumo_saude.index.astype(str)
cores_saude = sns.color_palette(n_colors=len(categorias_saude))

# Painel 1: média de Coffee_Intake
b1 = ax1.bar(categorias_saude, resumo_saude["Café (média)"], color=cores_saude)
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.set_title("Média de Coffee_Intake")
ax1.set_ylabel("Média")
ax1.set_ylim(0, resumo_saude["Café (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))

# Painel 2: média de BMI
b2 = ax2.bar(categorias_saude, resumo_saude["BMI (média)"], color=cores_saude)
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.set_title("Média de BMI")
ax2.set_ylabel("IMC médio")
ax2.set_ylim(0, resumo_saude["BMI (média)"].max() * (1 + MARGEM_TOPO_GRAFICO))

# Painel 3: distribuição percentual de Smoking
nao_fum = dist_smoking["Não fumante (0) %"]
fum = dist_smoking["Fumante (1) %"]
ax3.bar(categorias_saude, nao_fum, label="Não fumante (0)", color="#9ecae1")
b3_fum = ax3.bar(categorias_saude, fum, bottom=nao_fum, label="Fumante (1)", color="#de2d26")
ax3.bar_label(
    b3_fum,
    labels=[f"{v:.{CASAS_DECIMAIS}f}%" for v in fum],
    label_type="center",
    color="white",
    fontsize="small",
)
ax3.set_title("Distribuição de Smoking (%)")
ax3.set_ylabel("Percentual (%)")
ax3.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax3.legend(loc="upper right", fontsize="small")

for ax in (ax1, ax2, ax3):
    ax.set_xlabel("Health_Issues")
    ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

#### Resposta

- **Quantidade de registros (entre os `4.059` preenchidos):** `Mild` reúne **3.579** (`88,17%`), `Moderate` reúne **463** (`11,41%`) e `Severe` tem apenas **17** registros (`0,42%`).
- **Diferenças observadas:**
  - **`BMI` e `Heart_Rate`:** sobem de forma contínua conforme a gravidade — o `BMI` médio vai de **24,36** (`Mild`) para **27,26** (`Moderate`) e **31,98** (`Severe`), enquanto `Heart_Rate` passa de **70,80** para **71,34** e **72,94** bpm.
  - **`Smoking`:** a proporção de fumantes (`1`) cresce moderadamente de **19,22%** (`Mild`, com `80,78%` não fumantes) para **22,89%** (`Moderate`, `77,11%` não fumantes) e **23,53%** (`Severe`, `76,47%` não fumantes).
  - **`Coffee_Intake` e `Caffeine_mg`:** não seguem tendência crescente contínua — sobem levemente de `Mild` (**2,69** / **255,65 mg**) para `Moderate` (**2,80** / **266,47 mg**), mas **caem** em `Severe` (**2,49** / **236,06 mg**, abaixo até de `Mild`).
- **O café causa problemas de saúde?** **Não é possível afirmar causalidade**, por quatro razões:
  1. **Associação não é causalidade:** a análise exploratória apenas descreve padrões observados, sem desenho experimental nem controle de fatores externos.
  2. **Ausência de padrão crescente para o café:** o grupo `Severe` consome menos café em média do que `Mild` e `Moderate`.
  3. **Amostra muito pequena em `Severe` e alta proporção de ausentes:** `Severe` tem apenas `17` pessoas (qualquer mudança de 1 fumante altera o percentual em quase 6 pontos), e `59,41%` do dataset original está como `Não informado` em `Health_Issues`. Fatores como `BMI` e idade variam bem mais entre os grupos.
  4. **Dados sintéticos:** o conjunto foi gerado artificialmente para estudo e não representa evidência clínica.

## 7. LGPD e ética

Quais dados do dataset são pessoais ou sensíveis, quais riscos éticos o tratamento dessas informações traz e como o projeto se adequa aos princípios da LGPD (Lei nº 13.709/2018).

> O dataset é **sintético**: nenhum registro corresponde a uma pessoa real. Mesmo assim, a análise trata os dados como se fossem reais, porque a mesma estrutura de colunas, coletada de pessoas de verdade, estaria sujeita à LGPD.

### Classificação dos dados segundo a LGPD

- **Dado pessoal** (art. 5º, I): informação relacionada a pessoa natural identificada ou identificável.
- **Dado pessoal sensível** (art. 5º, II): entre outros, dado referente à saúde. Exige uma base legal mais restrita (art. 11).
- **Identificador direto:** aponta para um registro específico (`ID`).
- **Quase-identificador:** sozinho não identifica ninguém, mas combinado com outros pode identificar (`Age`, `Gender`, `Country`, `Occupation`).

In [ ]:
QUASE_IDENTIFICADORES = ["Age", "Gender", "Country", "Occupation"]

categorias_lgpd = {
    "ID": "Identificador direto",
    "Age": "Dado pessoal (quase-identificador)",
    "Gender": "Dado pessoal (quase-identificador)",
    "Country": "Dado pessoal (quase-identificador)",
    "Occupation": "Dado pessoal (quase-identificador)",
    "Coffee_Intake": "Dado pessoal (hábito)",
    "Caffeine_mg": "Dado pessoal (hábito)",
    "Physical_Activity_Hours": "Dado pessoal (hábito)",
    "Sleep_Hours": "Dado sensível (saúde)",
    "Sleep_Quality": "Dado sensível (saúde)",
    "Stress_Level": "Dado sensível (saúde)",
    "BMI": "Dado sensível (saúde)",
    "Heart_Rate": "Dado sensível (saúde)",
    "Health_Issues": "Dado sensível (saúde)",
    "Smoking": "Dado sensível (saúde)",
    "Alcohol_Consumption": "Dado sensível (saúde)",
}

classificacao_lgpd = pd.Series(categorias_lgpd).to_frame("Categoria LGPD")
classificacao_lgpd

In [ ]:
classificacao_lgpd["Categoria LGPD"].value_counts().to_frame("Quantidade de colunas")

### Risco de reidentificação

Remover o `ID` (feito no ETL) não basta para anonimizar. Quem conhece a idade, o gênero, o país e a ocupação de alguém pode encontrar o registro dessa pessoa se a combinação for rara, e com isso descobrir os dados de saúde dela.

A medida usada aqui é o **k-anonimato**: cada combinação de quase-identificadores deve aparecer em pelo menos `K_ANONIMATO` registros. Quanto menor o grupo, mais fácil reidentificar; um grupo de tamanho 1 identifica a pessoa.

In [ ]:
def medir_risco(dados, colunas, k=K_ANONIMATO):
    """Agrupa os registros pelas colunas informadas e mede quantos ficam em grupos menores que k."""
    tamanhos = dados.groupby(colunas, observed=True).size()
    em_risco = tamanhos[tamanhos < k].sum()
    # dtype object mantém as contagens como inteiros ao lado do percentual
    return pd.Series(dtype=object, data={
        "Registros": len(dados),
        "Combinações distintas": len(tamanhos),
        "Registros únicos na combinação": (tamanhos == 1).sum(),
        f"Registros em grupos com menos de {k}": em_risco,
        "% em risco": round(em_risco / len(dados) * 100, 2),
    })


medir_risco(df, QUASE_IDENTIFICADORES).to_frame("Idade exata")

### Medidas de proteção: generalização e supressão

Duas técnicas de anonimização aplicadas a uma versão do DataFrame destinada a compartilhamento:

1. **Generalização:** a idade exata é substituída por faixas etárias (limites definidos em `FAIXAS_ETARIAS`, no `.env`).
2. **Supressão:** os registros que, mesmo com as faixas, continuam em grupos com menos de `K_ANONIMATO` pessoas são retirados.

As análises do notebook continuam usando `df`, porque questões como a Q3 e a Q8 precisam da idade exata. `df_compartilhavel` é a versão que poderia sair do ambiente de análise.

> **Por que a última faixa vai de 45 a 80 anos:** há poucos registros acima de 60 anos (185). Com uma faixa separada de 60 a 80, todos eles caíam em grupos pequenos e a supressão excluía a faixa inteira, deixando os idosos fora dos dados compartilhados. Juntar as faixas reduz a precisão da idade, mas evita apagar um grupo da população.

In [ ]:
rotulos_faixas = [f"{inicio}–{fim - 1}" for inicio, fim in zip(FAIXAS_ETARIAS[:-1], FAIXAS_ETARIAS[1:])]
quase_identificadores_generalizados = ["Faixa_Etaria", "Gender", "Country", "Occupation"]

# Generalização: idade exata → faixa etária
df_generalizado = df.copy()
df_generalizado["Faixa_Etaria"] = pd.cut(
    df_generalizado["Age"], bins=FAIXAS_ETARIAS, labels=rotulos_faixas, right=False
)
df_generalizado = df_generalizado.drop(columns="Age")

# Supressão: retira os registros que continuam em grupos menores que K_ANONIMATO
tamanho_do_grupo = df_generalizado.groupby(
    quase_identificadores_generalizados, observed=True
)["Gender"].transform("size")
df_compartilhavel = df_generalizado[tamanho_do_grupo >= K_ANONIMATO].reset_index(drop=True)

pd.DataFrame({
    "Idade exata": medir_risco(df, QUASE_IDENTIFICADORES),
    "Faixa etária": medir_risco(df_generalizado, quase_identificadores_generalizados),
    "Faixa etária + supressão": medir_risco(df_compartilhavel, quase_identificadores_generalizados),
})

In [ ]:
df_compartilhavel["Faixa_Etaria"].value_counts(sort=False).to_frame("Registros")

### Riscos éticos

| Risco | Como aconteceria |
|---|---|
| **Reidentificação** | Cruzar idade, gênero, país e ocupação com outra fonte (rede social, cadastro de empresa) e chegar à pessoa e aos seus dados de saúde. |
| **Discriminação** | Seguradoras, planos de saúde ou empregadores usarem `BMI`, `Health_Issues`, `Smoking` ou `Alcohol_Consumption` para negar cobertura, cobrar mais ou recusar candidatos. |
| **Desvio de finalidade** | Dados coletados para um estudo sobre café e sono serem usados para publicidade ou perfilamento de consumo. |
| **Vazamento** | Acesso indevido ao arquivo expor dados sensíveis de milhares de pessoas. |
| **Conclusões causais indevidas** | Apresentar associações como causa ("café causa problemas de saúde") e estigmatizar grupos, como fumantes ou certas ocupações. |
| **Viés em modelos** | Um modelo como o da Q8 ser usado para decidir algo sobre uma pessoa, reproduzindo distorções da amostra (por exemplo, o grupo `Other` em `Gender` tem poucos registros). |

### Adequação aos princípios da LGPD

**Base legal.** Por envolver dados de saúde, um tratamento real exigiria uma das hipóteses do art. 11, como o consentimento específico e destacado do titular ou a realização de estudos por órgão de pesquisa, garantida, sempre que possível, a anonimização dos dados sensíveis. Dados anonimizados de forma irreversível deixam de ser considerados dados pessoais (art. 12).

| Princípio (art. 6º) | Como o projeto aplica |
|---|---|
| Finalidade | Os dados são usados só para a análise exploratória e acadêmica descrita no enunciado. |
| Adequação | As análises têm relação direta com o tema do estudo (café, sono, estresse e saúde). |
| Necessidade | Minimização: o `ID` é removido no ETL e a idade é generalizada na versão compartilhável. |
| Livre acesso e transparência | O README e o notebook documentam quais dados são usados, como e para quê. |
| Qualidade dos dados | O ETL verifica ausentes, duplicados e categorias sem alterar o arquivo original. |
| Segurança e prevenção | Repositório privado, acesso restrito aos integrantes e compartilhamento apenas de `df_compartilhavel` (art. 46). |
| Não discriminação | Os resultados são apresentados como associações em dados sintéticos, nunca como critério para decisões sobre pessoas. |
| Responsabilização | As decisões de tratamento ficam registradas no código, no histórico do Git e nos pull requests. |

### Resposta

- **Dados sensíveis:** 8 das 16 colunas são dados referentes à saúde (`Sleep_Hours`, `Sleep_Quality`, `Stress_Level`, `BMI`, `Heart_Rate`, `Health_Issues`, `Smoking` e `Alcohol_Consumption`). `ID` é identificador direto, e `Age`, `Gender`, `Country` e `Occupation` são quase-identificadores.
- **Remover o ID não anonimiza:** com a idade exata, 3.084 registros são únicos na combinação de quase-identificadores e 8.910 (89,1%) ficam em grupos com menos de 5 pessoas.
- **Generalização e supressão funcionam:** com as faixas etárias (18–29, 30–44 e 45–80), os registros em risco caem para 237 (2,37%); retirando esses registros, todos os grupos passam a ter pelo menos 5 pessoas e 9.763 registros (97,6%) são preservados, sem excluir nenhuma faixa etária.
- **Principais riscos éticos:** reidentificação, discriminação por seguradoras ou empregadores, desvio de finalidade, vazamento e conclusões causais indevidas.
- **Adequação à LGPD:** base legal do art. 11 para dados de saúde, anonimização sempre que possível (art. 12), minimização, segurança no armazenamento e transparência sobre o tratamento. Como o dataset é sintético, essas medidas demonstram como o projeto se comportaria com dados reais.

## 8. Hipóteses

Investigação de dez hipóteses (**H1 a H10**) sobre consumo de café, sono, estresse, indicadores fisiológicos, hábitos de vida e perfil demográfico. Cada hipótese segue a estrutura: **enunciado (com os dados utilizados) → análise em Python → leitura do resultado → conclusão**.

> Todas as análises partem de `df` (dados limpos no ETL da Q2) e avaliam associações no dataset **sintético**, sem inferir causalidade clínica.

In [ ]:
# Utilitários e rótulos compartilhados pelas hipóteses
ROTULOS_CAFE = ["0–1", ">1–3", ">3–5", ">5"]
ROTULOS_CAFEINA = ["0–100", ">100–200", ">200–300", ">300–400", ">400–500", ">500"]
ROTULOS_IMC = ["Abaixo (<18,5)", "Normal (18,5–24,9)", "Sobrepeso (25–29,9)", "Obesidade (>=30)"]
LIMITE_CAFE_ALTO = LIMITES_CAFE[-2]
LIMITE_OBESIDADE_IMC = LIMITES_IMC[-2]


def formatar_p(p):
    """Formata p-valor para leitura direta na análise das hipóteses."""
    if p < LIMITE_P_EXIBICAO:
        return f"< {LIMITE_P_EXIBICAO}"
    return f"{p:.{CASAS_CORRELACAO}f}"

### H1 — Mais café reduz a duração do sono

- **Enunciado:** Quanto maior o consumo diário de café (`Coffee_Intake`), menor tende a ser o tempo de sono (`Sleep_Hours`), mesmo controlando fatores demográficos e comportamentais (`Age`, `BMI`, `Physical_Activity_Hours`, `Smoking` e `Alcohol_Consumption`).
- **Dados utilizados:** `Coffee_Intake` (contínua e agrupada nas faixas `0–1`, `>1–3`, `>3–5` e `>5` via `LIMITES_CAFE`), `Sleep_Hours`, variáveis de controle (`Age`, `BMI`, `Physical_Activity_Hours`, `Smoking`, `Alcohol_Consumption`) e estratificação pelas faixas etárias de `FAIXAS_ETARIAS` (`18–29`, `30–44` e `45–80`).
- **Análise:** Correlações de Pearson e Spearman, regressão linear múltipla, média de sono por faixa de consumo e correlação estratificada por faixa etária.

In [ ]:
r_pearson_h1, p_pearson_h1 = stats.pearsonr(df["Coffee_Intake"], df["Sleep_Hours"])
r_spearman_h1, p_spearman_h1 = stats.spearmanr(df["Coffee_Intake"], df["Sleep_Hours"])
r2_h1 = (r_pearson_h1 ** 2) * 100

preditores_h1 = ["Coffee_Intake", "Age", "BMI", "Physical_Activity_Hours", "Smoking", "Alcohol_Consumption"]
reg_h1 = LinearRegression().fit(df[preditores_h1], df["Sleep_Hours"])
coef_cafe_h1 = reg_h1.coef_[0]
min_por_xicara = abs(coef_cafe_h1) * MINUTOS_POR_HORA

faixas_cafe_h1 = pd.cut(df["Coffee_Intake"], bins=LIMITES_CAFE, include_lowest=True, labels=ROTULOS_CAFE)
sono_por_cafe_h1 = (
    df.groupby(faixas_cafe_h1, observed=True)["Sleep_Hours"]
    .agg(Registros="count", **{"Sono médio (h)": "mean", "Sono mediano (h)": "median"})
    .round(CASAS_DECIMAIS)
)

faixas_idade_h1 = pd.cut(
    df["Age"],
    bins=FAIXAS_ETARIAS,
    labels=[f"{a}–{b - 1}" for a, b in zip(FAIXAS_ETARIAS[:-1], FAIXAS_ETARIAS[1:])],
    right=False,
)
corr_etaria_h1 = pd.DataFrame([
    {
        "Faixa etária": faixa,
        "Registros": len(sub),
        "Pearson r": round(stats.pearsonr(sub["Coffee_Intake"], sub["Sleep_Hours"])[0], CASAS_CORRELACAO),
    }
    for faixa, sub in df.groupby(faixas_idade_h1, observed=True)
]).set_index("Faixa etária")

print(f"Pearson r = {r_pearson_h1:.{CASAS_CORRELACAO}f} (p {formatar_p(p_pearson_h1)}) | R² = {r2_h1:.{CASAS_DECIMAIS}f}%")
print(f"Spearman rho = {r_spearman_h1:.{CASAS_CORRELACAO}f} (p {formatar_p(p_spearman_h1)})")
print(f"Coeficiente ajustado de Coffee_Intake: {coef_cafe_h1:.{CASAS_CORRELACAO}f} h/unidade (≈ -{min_por_xicara:.1f} min)")
display(sono_por_cafe_h1)
display(corr_etaria_h1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_h1 = sns.color_palette(n_colors=len(sono_por_cafe_h1))
b1 = ax1.bar(sono_por_cafe_h1.index.astype(str), sono_por_cafe_h1["Sono médio (h)"], color=cores_h1)
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.set_title("H1: Horas médias de sono por faixa de consumo")
ax1.set_xlabel("Consumo diário (Coffee_Intake)")
ax1.set_ylabel("Sono médio (h)")
ax1.set_ylim(0, sono_por_cafe_h1["Sono médio (h)"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.grid(axis="x", visible=False)

b2 = ax2.bar(corr_etaria_h1.index.astype(str), corr_etaria_h1["Pearson r"], color=sns.color_palette("muted", n_colors=len(corr_etaria_h1)))
ax2.bar_label(b2, fmt=f"%.{CASAS_CORRELACAO}f", padding=PADDING_ROTULO)
ax2.axhline(0, color="gray", linewidth=1)
ax2.set_title("H1: Correlação café × sono por faixa etária")
ax2.set_xlabel("Faixa etária (anos)")
ax2.set_ylabel("Correlação de Pearson (r)")
ax2.set_ylim(corr_etaria_h1["Pearson r"].min() * (1 + MARGEM_TOPO_GRAFICO), 0.02)
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** A correlação entre `Coffee_Intake` e `Sleep_Hours` é **r = -0,1903** (`p < 0,001`; Spearman **-0,1901**), explicando cerca de **3,62%** (`R²`) da variância das horas de sono. Na regressão controlada, cada unidade adicional de café associa-se a **-0,1604 h** (cerca de **10 minutos** a menos de sono). A média cai de **6,96 h** (`0–1`) para **6,12 h** (`>5`), e o efeito permanece estável em todas as faixas etárias (`r` entre **-0,1827** e **-0,1999**).
- **Conclusão:** **Hipótese confirmada.** O efeito negativo do café sobre a duração do sono é consistente em todas as idades e independente das variáveis de controle, embora tenha magnitude moderada (`R² = 3,62%`).

### H2 — Mais café piora a qualidade do sono

- **Enunciado:** Consumidores pesados de café apresentam proporção maior de sono `Poor` e menor de sono `Excellent` do que quem consome pouco café.
- **Dados utilizados:** `Coffee_Intake` agrupada nas faixas `0–1`, `>1–3`, `>3–5` e `>5` (`LIMITES_CAFE`) e a variável categórica ordinal `Sleep_Quality` (`Poor`, `Fair`, `Good`, `Excellent`).
- **Análise:** Tabela de distribuição percentual (`pd.crosstab` normalizada por linha) da qualidade do sono em cada faixa de consumo e gráfico de barras empilhadas 100%.

In [ ]:
faixas_cafe_h2 = pd.cut(df["Coffee_Intake"], bins=LIMITES_CAFE, include_lowest=True, labels=ROTULOS_CAFE)
dist_sono_h2 = (
    pd.crosstab(faixas_cafe_h2, df["Sleep_Quality"], normalize="index")
    .reindex(columns=ORDEM_SLEEP_QUALITY) * 100
).round(CASAS_DECIMAIS)

razao_poor = dist_sono_h2.loc[">5", "Poor"] / dist_sono_h2.loc["0–1", "Poor"]
print(f"Proporção de sono Poor sobe {razao_poor:.{CASAS_DECIMAIS}f}x da faixa 0–1 ({dist_sono_h2.loc['0–1', 'Poor']:.{CASAS_DECIMAIS}f}%) para >5 ({dist_sono_h2.loc['>5', 'Poor']:.{CASAS_DECIMAIS}f}%)")
display(dist_sono_h2)

ax = dist_sono_h2.plot(kind="bar", stacked=True, figsize=(FIG_LARGURA, FIG_ALTURA), colormap="RdYlGn")
ax.set_title("H2: Distribuição de Sleep_Quality (%) por faixa de consumo de café")
ax.set_xlabel("Faixa de consumo (Coffee_Intake)")
ax.set_ylabel("Percentual (%)")
ax.set_xticklabels(dist_sono_h2.index.astype(str), rotation=0)
ax.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax.legend(loc="upper center", ncol=4)
ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** A proporção de sono `Poor` cresce continuamente de **5,64%** (`0–1`) para **8,52%** (`>1–3`), **12,23%** (`>3–5`) e **18,32%** (`>5`) — um aumento de **3,25 vezes** (`≈ 3,3x`). No sentido oposto, o sono `Excellent` cai monotonicamente de **20,27%** (`0–1`) para **14,42%**, **9,46%** e **6,03%** (`>5`).
- **Conclusão:** **Hipótese confirmada**, com gradiente dose-resposta claro e sem inversões entre todas as faixas de consumo.

### H3 — Mais café está associado a mais estresse

- **Enunciado:** Pessoas classificadas com nível de estresse mais alto (`High`) consomem mais café em média do que aquelas com estresse `Medium` e `Low`; além disso, avalia-se se `Stress_Level` traz informação independente de `Sleep_Quality`.
- **Dados utilizados:** `Coffee_Intake` (consumo diário), `Stress_Level` (`Low`, `Medium`, `High`) e `Sleep_Quality` (`Poor`, `Fair`, `Good`, `Excellent`).
- **Análise:** Médias de consumo por nível de estresse, ANOVA *one-way* (`f_oneway`) e tabela cruzada (`pd.crosstab`) entre `Stress_Level` e `Sleep_Quality`.

In [ ]:
resumo_estresse_h3 = (
    df.groupby("Stress_Level", observed=True)["Coffee_Intake"]
    .agg(Registros="count", Média="mean", Mediana="median")
    .round(CASAS_DECIMAIS)
)
grupos_h3 = [sub["Coffee_Intake"].values for _, sub in df.groupby("Stress_Level", observed=True)]
f_h3, p_h3 = stats.f_oneway(*grupos_h3)

cruzamento_h3 = pd.crosstab(df["Stress_Level"], df["Sleep_Quality"])

print(f"ANOVA one-way (Coffee_Intake por Stress_Level): F = {f_h3:.{CASAS_DECIMAIS}f}, p {formatar_p(p_h3)}")
display(resumo_estresse_h3)
display(cruzamento_h3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
b1 = ax1.bar(resumo_estresse_h3.index.astype(str), resumo_estresse_h3["Média"], color=sns.color_palette(n_colors=len(resumo_estresse_h3)))
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.set_title(f"H3: Consumo médio por estresse (F = {f_h3:.{CASAS_DECIMAIS}f})")
ax1.set_xlabel("Stress_Level")
ax1.set_ylabel("Média de Coffee_Intake")
ax1.set_ylim(0, resumo_estresse_h3["Média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.grid(axis="x", visible=False)

sns.heatmap(cruzamento_h3, annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=1, linecolor="white", ax=ax2)
ax2.set_title("H3: Cruzamento exato Stress_Level × Sleep_Quality")
ax2.set_xlabel("Sleep_Quality")
ax2.set_ylabel("Stress_Level")

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** O consumo médio sobe de **2,37** (`Low`) para **2,76** (`Medium`) e **2,98** (`High`), com **F = 117,09** e **p < 0,001**. Porém, a tabela cruzada revela que `Stress_Level` é uma **recodificação determinística** de `Sleep_Quality`: `High` coincide 100% com `Poor` (`961`), `Medium` coincide 100% com `Fair` (`2.050`), e `Low` reúne 100% de `Good` (`5.637`) e `Excellent` (`1.352`).
- **Conclusão:** **Hipótese confirmada estatisticamente, com ressalva crítica:** neste dataset, `Stress_Level` e `Sleep_Quality` carregam exatamente a mesma informação, não permitindo tratá-los como evidências independentes nem separar estresse de sono ruim.

### H4 — Mais café eleva a frequência cardíaca

- **Enunciado:** Maior consumo de café está associado a batimentos cardíacos mais altos (`Heart_Rate`) e maior proporção de frequência cardíaca elevada (`Heart_Rate >= LIMITE_FC_ALTA` bpm).
- **Dados utilizados:** `Coffee_Intake` (contínua, em faixas `LIMITES_CAFE`, e corte `>= LIMITE_CAFE_ALTO`) e `Heart_Rate` (em bpm, além do indicador booleano `Heart_Rate >= 85` bpm).
- **Análise:** Correlação de Pearson, regressão linear simples (`linregress`), média de `Heart_Rate` por faixa de consumo e comparação da proporção com FC ≥ 85 bpm (`>= 5` vs. `< 5`).

In [ ]:
r_h4, p_h4 = stats.pearsonr(df["Coffee_Intake"], df["Heart_Rate"])
reg_h4 = stats.linregress(df["Coffee_Intake"], df["Heart_Rate"])

faixas_cafe_h4 = pd.cut(df["Coffee_Intake"], bins=LIMITES_CAFE, include_lowest=True, labels=ROTULOS_CAFE)
fc_alta = df["Heart_Rate"] >= LIMITE_FC_ALTA
resumo_fc_h4 = (
    df.assign(FC_Alta=fc_alta * 100)
    .groupby(faixas_cafe_h4, observed=True)
    .agg(Registros=("Heart_Rate", "count"), **{"FC média (bpm)": ("Heart_Rate", "mean"), f"FC >= {LIMITE_FC_ALTA} (%)": ("FC_Alta", "mean")})
    .round(CASAS_DECIMAIS)
)

prop_5_mais = fc_alta[df["Coffee_Intake"] >= LIMITE_CAFE_ALTO].mean() * 100
prop_demais = fc_alta[df["Coffee_Intake"] < LIMITE_CAFE_ALTO].mean() * 100

print(f"Pearson r = {r_h4:.{CASAS_CORRELACAO}f} (p {formatar_p(p_h4)}) | Inclinação = +{reg_h4.slope:.{CASAS_DECIMAIS}f} bpm por unidade de café")
print(f"FC >= {LIMITE_FC_ALTA} bpm: {prop_5_mais:.{CASAS_DECIMAIS}f}% (>= {int(LIMITE_CAFE_ALTO)}) contra {prop_demais:.{CASAS_DECIMAIS}f}% (< {int(LIMITE_CAFE_ALTO)})")
display(resumo_fc_h4)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
cores_h4 = sns.color_palette(n_colors=len(resumo_fc_h4))
b1 = ax1.bar(resumo_fc_h4.index.astype(str), resumo_fc_h4["FC média (bpm)"], color=cores_h4)
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.set_title("H4: Frequência cardíaca média por faixa")
ax1.set_xlabel("Faixa de consumo (Coffee_Intake)")
ax1.set_ylabel("FC média (bpm)")
ax1.set_ylim(0, resumo_fc_h4["FC média (bpm)"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.grid(axis="x", visible=False)

col_fc_alta = f"FC >= {LIMITE_FC_ALTA} (%)"
b2 = ax2.bar(resumo_fc_h4.index.astype(str), resumo_fc_h4[col_fc_alta], color=cores_h4)
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax2.set_title(f"H4: Proporção com FC >= {LIMITE_FC_ALTA} bpm")
ax2.set_xlabel("Faixa de consumo (Coffee_Intake)")
ax2.set_ylabel("Registros (%)")
ax2.set_ylim(0, resumo_fc_h4[col_fc_alta].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** A correlação é **r = 0,0601** (`p < 0,001`), com inclinação de **+0,41 bpm** por unidade de café. A FC média passa de **69,50 bpm** (`0–1`) para **71,91 bpm** (`>5`) — variação de **2,41 bpm** — e a proporção com `FC >= 85 bpm` vai de **8,01%** (`< 5`) para **11,80%** (`>= 5`; chegando a `12,50%` acima de `5`).
- **Conclusão:** **Hipótese confirmada estatisticamente, mas de efeito pequeno.** Devido ao tamanho amostral (`10.000`), o p-valor é muito baixo, porém a diferença prática (~2,4 bpm entre extremos) é modesta.

### H5 — Ultrapassar 400 mg de cafeína por dia marca uma piora clara

- **Enunciado:** Acima de `LIMITE_CAFEINA_MG` (`400 mg/dia`, referência comumente citada como limite diário seguro), os indicadores de sono, frequência cardíaca e problemas de saúde pioram de forma abrupta (efeito de limiar) ou gradual?
- **Dados utilizados:** `Caffeine_mg` (dividida em `> 400 mg` vs. `<= 400 mg`, e em faixas contínuas `LIMITES_CAFEINA`), `Sleep_Hours`, `Heart_Rate`, `Health_Issues` (presença de problema quando diferente de `ROTULO_NAO_INFORMADO`) e `Coffee_Intake`.
- **Análise:** Comparação direta dos grupos `> 400 mg` e `<= 400 mg` (testes *t* e qui-quadrado) combinada à curva contínua por faixas de cafeína para testar se existe salto em `400 mg`.

In [ ]:
acima_400 = df["Caffeine_mg"] > LIMITE_CAFEINA_MG
tem_problema_h5 = df["Health_Issues"] != ROTULO_NAO_INFORMADO

grupo_label = acima_400.map({False: f"<= {int(LIMITE_CAFEINA_MG)} mg", True: f"> {int(LIMITE_CAFEINA_MG)} mg"})
comp_400 = (
    df.assign(Tem_Problema=tem_problema_h5 * 100)
    .groupby(grupo_label)
    .agg(
        Registros=("Caffeine_mg", "count"),
        **{"Sono médio (h)": ("Sleep_Hours", "mean"), "FC média (bpm)": ("Heart_Rate", "mean"), "Problemas de saúde (%)": ("Tem_Problema", "mean")},
    )
    .assign(**{"% da amostra": lambda t: t["Registros"] / len(df) * 100})
    .round(CASAS_DECIMAIS)
)

_, p_sono_h5 = stats.ttest_ind(df.loc[acima_400, "Sleep_Hours"], df.loc[~acima_400, "Sleep_Hours"])
_, p_fc_h5 = stats.ttest_ind(df.loc[acima_400, "Heart_Rate"], df.loc[~acima_400, "Heart_Rate"])
_, p_saude_h5, _, _ = stats.chi2_contingency(pd.crosstab(acima_400, tem_problema_h5))

faixas_caf_h5 = pd.cut(df["Caffeine_mg"], bins=LIMITES_CAFEINA, include_lowest=True, labels=ROTULOS_CAFEINA)
curva_caf_h5 = (
    df.assign(Tem_Problema=tem_problema_h5 * 100)
    .groupby(faixas_caf_h5, observed=True)
    .agg(Registros=("Caffeine_mg", "count"), **{"Sono médio (h)": ("Sleep_Hours", "mean"), "Problemas de saúde (%)": ("Tem_Problema", "mean")})
    .round(CASAS_DECIMAIS)
)

print(f"p-valores (>400 vs <=400 mg): Sono p {formatar_p(p_sono_h5)} | FC p {formatar_p(p_fc_h5)} | Saúde p {formatar_p(p_saude_h5)}")
display(comp_400)
display(curva_caf_h5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
x_caf = range(len(curva_caf_h5))
ax1.plot(x_caf, curva_caf_h5["Sono médio (h)"], marker="o", linewidth=2)
ax1.axvline(3.5, color="crimson", linestyle="--", label=f"Corte {int(LIMITE_CAFEINA_MG)} mg")
ax1.set_xticks(list(x_caf))
ax1.set_xticklabels(curva_caf_h5.index.astype(str), rotation=20)
ax1.set_title("H5: Sono médio ao longo das faixas de cafeína")
ax1.set_xlabel("Cafeína (mg/dia)")
ax1.set_ylabel("Sono médio (h)")
ax1.legend(loc="upper right")

ax2.plot(x_caf, curva_caf_h5["Problemas de saúde (%)"], marker="s", color="darkorange", linewidth=2)
ax2.axvline(3.5, color="crimson", linestyle="--", label=f"Corte {int(LIMITE_CAFEINA_MG)} mg")
ax2.set_xticks(list(x_caf))
ax2.set_xticklabels(curva_caf_h5.index.astype(str), rotation=20)
ax2.set_title("H5: Problemas de saúde (%) por faixa de cafeína")
ax2.set_xlabel("Cafeína (mg/dia)")
ax2.set_ylabel("Com problemas de saúde (%)")
ax2.legend(loc="upper left")

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** O grupo `> 400 mg` (`1.246` registros; **12,46%**) dorme menos (**6,24 h** vs. **6,69 h**, `p < 0,001`), tem FC ligeiramente maior (**71,48** vs. **70,49 bpm**, `p < 0,001`) e maior taxa de problemas de saúde (**49,68%** vs. **39,30%**, `p < 0,001`). Porém, as curvas por faixa de `Caffeine_mg` variam de forma contínua e progressiva, sem salto abrupto na marca de `400 mg`.
- **Conclusão:** **Hipótese parcialmente confirmada.** Quem ultrapassa `400 mg/dia` apresenta indicadores piores, mas o comportamento reflete um efeito contínuo (não um limiar rígido) — lembrando ainda da Q5 que `Caffeine_mg` e `Coffee_Intake` são praticamente redundantes (`r = 0,9998`).

### H6 — O café afeta a saúde apenas indiretamente, via sono

- **Enunciado:** A associação aparente entre maior consumo de café e presença de problemas de saúde desaparece quando se controla (estratifica) pela qualidade do sono (`Sleep_Quality`).
- **Dados utilizados:** `Coffee_Intake` (corte em `LIMITE_CORTE_CAFE = 3` xícaras: `< 3` vs. `>= 3`, além da comparação entre extremos `0–1` e `>5`), `Sleep_Quality` (`Poor`, `Fair`, `Good`, `Excellent`) e indicador de problema em `Health_Issues` (`!= ROTULO_NAO_INFORMADO`).
- **Análise:** Taxa de problemas de saúde por nível de consumo na amostra geral e estratificada dentro de cada categoria de `Sleep_Quality`.

In [ ]:
tem_problema_h6 = (df["Health_Issues"] != ROTULO_NAO_INFORMADO) * 100
consumo_grupo_h6 = pd.Series(
    [f">= {int(LIMITE_CORTE_CAFE)}" if v >= LIMITE_CORTE_CAFE else f"< {int(LIMITE_CORTE_CAFE)}" for v in df["Coffee_Intake"]],
    index=df.index,
)

faixas_ext_h6 = pd.cut(df["Coffee_Intake"], bins=LIMITES_CAFE, include_lowest=True, labels=ROTULOS_CAFE)
taxa_extremos_h6 = tem_problema_h6.groupby(faixas_ext_h6, observed=True).mean().round(CASAS_DECIMAIS)
taxa_geral_h6 = tem_problema_h6.groupby(consumo_grupo_h6).mean().round(CASAS_DECIMAIS)

taxa_estratificada_h6 = (
    df.assign(Tem_Problema=tem_problema_h6, Consumo=consumo_grupo_h6)
    .pivot_table(index="Sleep_Quality", columns="Consumo", values="Tem_Problema", aggfunc="mean", observed=True)
    .reindex(ORDEM_SLEEP_QUALITY)[[f"< {int(LIMITE_CORTE_CAFE)}", f">= {int(LIMITE_CORTE_CAFE)}"]]
    .round(CASAS_DECIMAIS)
)

print(f"Sem estratificação (extremos): 0–1 = {taxa_extremos_h6['0–1']:.{CASAS_DECIMAIS}f}% | >5 = {taxa_extremos_h6['>5']:.{CASAS_DECIMAIS}f}%")
print(f"Sem estratificação (corte {int(LIMITE_CORTE_CAFE)}): < 3 = {taxa_geral_h6['< 3']:.{CASAS_DECIMAIS}f}% | >= 3 = {taxa_geral_h6['>= 3']:.{CASAS_DECIMAIS}f}%")
display(taxa_estratificada_h6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
b1 = ax1.bar(taxa_extremos_h6.index.astype(str), taxa_extremos_h6.values, color=sns.color_palette(n_colors=len(taxa_extremos_h6)))
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax1.set_title("H6: Problemas de saúde sem controle do sono")
ax1.set_xlabel("Faixa de Coffee_Intake")
ax1.set_ylabel("Com problemas de saúde (%)")
ax1.set_ylim(0, taxa_extremos_h6.max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.grid(axis="x", visible=False)

taxa_estratificada_h6.plot(kind="bar", ax=ax2)
ax2.set_title("H6: Taxa de problemas dentro de cada Sleep_Quality")
ax2.set_xlabel("Sleep_Quality")
ax2.set_ylabel("Com problemas de saúde (%)")
ax2.set_xticklabels(taxa_estratificada_h6.index.astype(str), rotation=0)
ax2.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax2.legend(title="Coffee_Intake", loc="upper right")
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** Sem controle do sono, a taxa de problemas de saúde sobe de **33,20%** (`0–1`) para **55,39%** (`>5`) — ou de **37,11%** (`< 3`) para **46,44%** (`>= 3`). Porém, ao estratificar por `Sleep_Quality`, a diferença praticamente desaparece: em `Good`, fica em **14,92%** (`< 3`) vs. **14,17%** (`>= 3`); em `Excellent`, **16,67%** vs. **15,68%**; e em `Fair` e `Poor` chega a **100,00%** em ambos os grupos.
- **Conclusão:** **Hipótese confirmada.** Neste dataset sintético, o consumo de café não apresenta efeito direto sobre os problemas de saúde — toda a associação aparente é mediada pela piora do sono.

### H7 — Após os 50 anos, o risco de problemas de saúde dispara

- **Enunciado:** A idade (`Age`) atua como fator determinante para apresentar problemas de saúde e agravar sua severidade, com salto expressivo após `CORTE_IDADE_RISCO` (`50` anos), independentemente do consumo de café ou do sono.
- **Dados utilizados:** `Age` (comparação `<= 50` vs. `> 50` e por idade individual), `Health_Issues` (`Não informado`, `Mild`, `Moderate`, `Severe`), `Coffee_Intake` e `Sleep_Hours`.
- **Análise:** Taxa de problemas de saúde antes e após os 50 anos, média de idade por grau de severidade e correlação da idade com café e sono.

In [ ]:
tem_problema_h7 = (df["Health_Issues"] != ROTULO_NAO_INFORMADO) * 100
grupo_idade_h7 = pd.Series(
    [f"> {CORTE_IDADE_RISCO} anos" if idade > CORTE_IDADE_RISCO else f"<= {CORTE_IDADE_RISCO} anos" for idade in df["Age"]],
    index=df.index,
)
taxa_corte_idade = tem_problema_h7.groupby(grupo_idade_h7).agg(Registros="count", **{"Problemas de saúde (%)": "mean"}).round(CASAS_DECIMAIS)
taxa_por_idade = tem_problema_h7.groupby(df["Age"]).mean().round(CASAS_DECIMAIS)

idade_por_severidade = (
    df.groupby("Health_Issues", observed=True)["Age"]
    .agg(Registros="count", **{"Idade média": "mean", "Idade mediana": "median"})
    .reindex([ROTULO_NAO_INFORMADO] + ORDEM_HEALTH_ISSUES)
    .round(CASAS_DECIMAIS)
)

r_id_cafe, p_id_cafe = stats.pearsonr(df["Age"], df["Coffee_Intake"])
r_id_sono, p_id_sono = stats.pearsonr(df["Age"], df["Sleep_Hours"])

print(f"Correlação Age × Coffee_Intake: r = {r_id_cafe:.{CASAS_CORRELACAO}f} (p {formatar_p(p_id_cafe)})")
print(f"Correlação Age × Sleep_Hours:   r = {r_id_sono:.{CASAS_CORRELACAO}f} (p {formatar_p(p_id_sono)})")
print(f"Taxa aos {CORTE_IDADE_RISCO} anos: {taxa_por_idade.loc[CORTE_IDADE_RISCO]:.{CASAS_DECIMAIS}f}% | Aos {CORTE_IDADE_RISCO + 1} anos: {taxa_por_idade.loc[CORTE_IDADE_RISCO + 1]:.{CASAS_DECIMAIS}f}%")
display(taxa_corte_idade)
display(idade_por_severidade)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
ax1.plot(taxa_por_idade.index, taxa_por_idade.values, marker="o", markersize=4)
ax1.axvline(CORTE_IDADE_RISCO, color="crimson", linestyle="--", label=f"Corte em {CORTE_IDADE_RISCO} anos")
ax1.set_title("H7: Taxa de problemas de saúde por idade exata")
ax1.set_xlabel("Idade (anos)")
ax1.set_ylabel("Com problemas de saúde (%)")
ax1.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="lower right")

b2 = ax2.bar(idade_por_severidade.index.astype(str), idade_por_severidade["Idade média"], color=sns.color_palette(n_colors=len(idade_por_severidade)))
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.set_title("H7: Idade média por categoria de Health_Issues")
ax2.set_xlabel("Health_Issues")
ax2.set_ylabel("Idade média (anos)")
ax2.set_ylim(0, idade_por_severidade["Idade média"].max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** Até `50` anos, a taxa agregada de problemas de saúde é de **34,55%** (oscilando perto de `30–38%` por idade; **30,66%** aos `50` anos). A partir de **51 anos**, atinge **100,00%** sem exceções. A severidade acompanha a idade: média de **32,86 anos** (`Não informado`), **36,62** (`Mild`), **48,09** (`Moderate`) e **56,59** (`Severe`). Enquanto isso, `Age` não tem relação linear relevante com `Coffee_Intake` (`r = -0,0122`) nem com `Sleep_Hours` (`r = 0,0050`).
- **Conclusão:** **Hipótese confirmada**, revelando uma regra de limiar rígido logo após os `50` anos no dataset sintético (e não uma progressão linear suave).

### H8 — Obesidade (IMC ≥ 30) praticamente garante problemas de saúde

- **Enunciado:** O Índice de Massa Corporal (`BMI`) possui efeito de limiar a partir da obesidade (`BMI >= LIMITE_OBESIDADE_IMC`), elevando drasticamente a chance de problemas de saúde; além disso, avalia-se a combinação dos três fatores de proteção (`Age <= 50`, `BMI < 30` e sono `Good`/`Excellent`).
- **Dados utilizados:** `BMI` (arredondado para inteiro na faixa `FAIXA_IMC_TABELA` e categorizado via `LIMITES_IMC`), `Health_Issues`, `Age` e `Sleep_Quality`.
- **Análise:** Taxa de problemas de saúde por valor arredondado de IMC e por categoria clínica, seguida da verificação da taxa no subgrupo sem nenhum dos três fatores de risco.

In [ ]:
tem_problema_h8 = (df["Health_Issues"] != ROTULO_NAO_INFORMADO) * 100
bmi_cat_h8 = pd.cut(df["BMI"], bins=LIMITES_IMC, right=False, labels=ROTULOS_IMC)
taxa_cat_bmi = (
    df.assign(Tem_Problema=tem_problema_h8)
    .groupby(bmi_cat_h8, observed=True)["Tem_Problema"]
    .agg(Registros="count", **{"Problemas de saúde (%)": "mean"})
    .round(CASAS_DECIMAIS)
)

bmi_arred = df["BMI"].round(0).astype(int)
taxa_bmi_arred = (
    df.assign(Tem_Problema=tem_problema_h8)
    .groupby(bmi_arred)["Tem_Problema"]
    .agg(Registros="count", **{"Problemas de saúde (%)": "mean"})
    .loc[FAIXA_IMC_TABELA[0]:FAIXA_IMC_TABELA[1]]
    .round(CASAS_DECIMAIS)
)

sem_risco_h8 = (df["Age"] <= CORTE_IDADE_RISCO) & (df["BMI"] < LIMITE_OBESIDADE_IMC) & (df["Sleep_Quality"].isin(["Good", "Excellent"]))
taxa_sem_risco = tem_problema_h8[sem_risco_h8].mean()
taxa_bmi_30_5 = tem_problema_h8[df["BMI"] >= LIMITE_OBESIDADE_IMC + 0.5].mean()

print(f"IMC >= {LIMITE_OBESIDADE_IMC + 0.5}: {taxa_bmi_30_5:.{CASAS_DECIMAIS}f}% com problemas de saúde")
print(f"Subgrupo (Age <= {CORTE_IDADE_RISCO}, BMI < {int(LIMITE_OBESIDADE_IMC)}, sono Good/Excellent): n = {sem_risco_h8.sum()} ({sem_risco_h8.mean() * 100:.{CASAS_DECIMAIS}f}%), problemas = {taxa_sem_risco:.{CASAS_DECIMAIS}f}%")
display(taxa_cat_bmi)
display(taxa_bmi_arred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
b1 = ax1.bar(taxa_bmi_arred.index.astype(str), taxa_bmi_arred["Problemas de saúde (%)"], color="steelblue")
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO, fontsize="x-small")
ax1.set_title("H8: Taxa de problemas por IMC arredondado")
ax1.set_xlabel("IMC arredondado")
ax1.set_ylabel("Com problemas de saúde (%)")
ax1.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax1.grid(axis="x", visible=False)

rotulos_imc_plot = [rotulo.replace(" (", "\n(") for rotulo in taxa_cat_bmi.index.astype(str)]
b2 = ax2.bar(rotulos_imc_plot, taxa_cat_bmi["Problemas de saúde (%)"], color=sns.color_palette(n_colors=len(taxa_cat_bmi)))
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax2.set_title("H8: Taxa de problemas por categoria de IMC")
ax2.set_xlabel("Categoria de IMC")
ax2.set_ylabel("Com problemas de saúde (%)")
ax2.set_ylim(0, 100 * (1 + MARGEM_TOPO_GRAFICO))
ax2.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** Entre as categorias `Abaixo (<18,5)`, `Normal` e `Sobrepeso`, a taxa permanece entre **36,54%** e **37,24%** (entre IMC arredondado `25` e `29`, oscila de `33,77%` a `39,51%`). Ao atingir IMC arredondado `30`, salta para **64,13%**, e a partir de `31` (`BMI >= 30,5`) alcança **100,00%** (na categoria `Obesidade (>=30)` como um todo, é de **97,82%**). Combinando com a H7 e H6, o grupo com `Age <= 50`, `BMI < 30` e sono `Good`/`Excellent` (`5.927` pessoas; **59,27%** do dataset) registra exatamente **0,00%** de problemas de saúde.
- **Conclusão:** **Hipótese confirmada**, evidenciando um limiar nítido na entrada da obesidade (`IMC ≈ 30–31`). Os três limiares (idade > 50, IMC ≥ 30 e sono `Fair`/`Poor`) explicam praticamente toda a regra de geração de `Health_Issues` no dataset sintético.

### H9 — Tabagismo, álcool e atividade física influenciam a saúde

- **Enunciado:** Hábitos de vida como tabagismo (`Smoking`), consumo de álcool (`Alcohol_Consumption`) e horas de atividade física (`Physical_Activity_Hours`) influenciam a ocorrência de problemas de saúde, a duração do sono, o IMC e a frequência cardíaca.
- **Dados utilizados:** `Smoking` (`0`/`1`), `Alcohol_Consumption` (`0`/`1`), `Physical_Activity_Hours`, `Health_Issues`, `Sleep_Hours`, `BMI` e `Heart_Rate`.
- **Análise:** Testes qui-quadrado de independência para tabagismo e álcool vs. problemas de saúde, ANOVA da atividade física entre categorias de `Health_Issues`, correlações da atividade física com `BMI`, `Heart_Rate` e `Sleep_Hours`, e teste *t* dos hábitos sobre `Sleep_Hours`.

In [ ]:
tem_problema_h9 = df["Health_Issues"] != ROTULO_NAO_INFORMADO

resultados_habitos = []
for col, nome in [("Smoking", "Tabagismo"), ("Alcohol_Consumption", "Consumo de álcool")]:
    _, p_saude, _, _ = stats.chi2_contingency(pd.crosstab(df[col], tem_problema_h9))
    _, p_sono = stats.ttest_ind(df.loc[df[col] == 1, "Sleep_Hours"], df.loc[df[col] == 0, "Sleep_Hours"])
    resultados_habitos.append({
        "Hábito": nome,
        "Problemas (Sim=1) %": round(tem_problema_h9[df[col] == 1].mean() * 100, CASAS_DECIMAIS),
        "Problemas (Não=0) %": round(tem_problema_h9[df[col] == 0].mean() * 100, CASAS_DECIMAIS),
        "p-valor Saúde": formatar_p(p_saude),
        "Sono (Sim=1) h": round(df.loc[df[col] == 1, "Sleep_Hours"].mean(), CASAS_DECIMAIS),
        "Sono (Não=0) h": round(df.loc[df[col] == 0, "Sleep_Hours"].mean(), CASAS_DECIMAIS),
        "p-valor Sono": formatar_p(p_sono),
    })

tabela_habitos_h9 = pd.DataFrame(resultados_habitos).set_index("Hábito")
f_ativ_h9, p_ativ_h9 = stats.f_oneway(*[sub["Physical_Activity_Hours"].values for _, sub in df.groupby("Health_Issues", observed=True)])
r_ativ_bmi, _ = stats.pearsonr(df["Physical_Activity_Hours"], df["BMI"])
r_ativ_fc, _ = stats.pearsonr(df["Physical_Activity_Hours"], df["Heart_Rate"])
r_ativ_sono, _ = stats.pearsonr(df["Physical_Activity_Hours"], df["Sleep_Hours"])

print(f"ANOVA Atividade Física entre grupos de Health_Issues: F = {f_ativ_h9:.{CASAS_DECIMAIS}f}, p = {formatar_p(p_ativ_h9)}")
print(f"Correlações de Atividade Física: BMI r = {r_ativ_bmi:.{CASAS_CORRELACAO}f} | FC r = {r_ativ_fc:.{CASAS_CORRELACAO}f} | Sono r = {r_ativ_sono:.{CASAS_CORRELACAO}f}")
display(tabela_habitos_h9)

fig, ax = plt.subplots(figsize=(FIG_LARGURA, FIG_ALTURA))
tabela_plot_h9 = tabela_habitos_h9[["Problemas (Não=0) %", "Problemas (Sim=1) %"]]
tabela_plot_h9.plot(kind="bar", ax=ax)
for cont in ax.containers:
    ax.bar_label(cont, fmt=f"%.{CASAS_DECIMAIS}f%%", padding=PADDING_ROTULO)
ax.set_title("H9: Taxa de problemas de saúde por tabagismo e consumo de álcool")
ax.set_xlabel("Hábito")
ax.set_ylabel("Com problemas de saúde (%)")
ax.set_xticklabels(tabela_plot_h9.index, rotation=0)
ax.set_ylim(0, tabela_plot_h9.max().max() * (1 + 2 * MARGEM_TOPO_GRAFICO))
ax.legend(loc="upper center", ncol=2)
ax.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** Não há diferença estatisticamente significativa: problemas de saúde ocorrem em **39,82%** dos fumantes contra **40,78%** dos não fumantes (`p = 0,4477`), e em **39,81%** de quem consome álcool contra **40,93%** de quem não consome (`p = 0,3062`). A atividade física não varia entre grupos de `Health_Issues` (`F = 0,89`, `p = 0,4451`) e tem correlação próxima de zero com `BMI` (`r = 0,0020`), `Heart_Rate` (`r = -0,0029`) e `Sleep_Hours` (`r = -0,0112`). Nenhum dos hábitos altera o sono.
- **Conclusão:** **Hipótese rejeitada.** Esse achado contraria o conhecimento clínico real e evidencia que o gerador sintético não modelou efeitos de tabagismo, álcool ou atividade física sobre saúde, sono ou IMC; portanto, este dataset não deve ser usado para inferências sobre esses hábitos.

### H10 — O perfil demográfico explica as diferenças de consumo de café

- **Enunciado:** Variáveis demográficas como país (`Country`), gênero (`Gender`) e ocupação (`Occupation`) explicam variações relevantes no consumo diário de café (`Coffee_Intake`), testando também se os problemas de saúde variam entre países.
- **Dados utilizados:** `Country` (20 países), `Gender` (3 categorias), `Occupation` (5 categorias), `Coffee_Intake` e indicador de problema em `Health_Issues`.
- **Análise:** ANOVA *one-way* de `Coffee_Intake` para cada variável demográfica (`Country`, `Gender`, `Occupation`) e teste qui-quadrado entre `Country` e presença de problemas de saúde.

In [ ]:
resumo_demog_h10 = []
for col, rotulo in [("Country", "País (20)"), ("Gender", "Gênero (3)"), ("Occupation", "Ocupação (5)")]:
    medias = df.groupby(col)["Coffee_Intake"].mean()
    f_val, p_val = stats.f_oneway(*[sub["Coffee_Intake"].values for _, sub in df.groupby(col)])
    resumo_demog_h10.append({
        "Dimensão": rotulo,
        "Menor média": round(medias.min(), CASAS_DECIMAIS),
        "Maior média": round(medias.max(), CASAS_DECIMAIS),
        "F (ANOVA)": round(f_val, CASAS_DECIMAIS),
        "p-valor": formatar_p(p_val),
    })

tabela_demog_h10 = pd.DataFrame(resumo_demog_h10).set_index("Dimensão")

tem_problema_h10 = df["Health_Issues"] != ROTULO_NAO_INFORMADO
_, p_pais_saude, _, _ = stats.chi2_contingency(pd.crosstab(df["Country"], tem_problema_h10))
taxas_pais = (tem_problema_h10.groupby(df["Country"]).mean() * 100).round(CASAS_DECIMAIS).sort_values()

print(f"Problemas de saúde por Country (qui-quadrado): p = {formatar_p(p_pais_saude)} | Menor: {taxas_pais.index[0]} ({taxas_pais.iloc[0]:.2f}%) | Maior: {taxas_pais.index[-1]} ({taxas_pais.iloc[-1]:.2f}%)")
display(tabela_demog_h10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_LARGURA, FIG_ALTURA))
medias_gen = df.groupby("Gender")["Coffee_Intake"].mean().round(CASAS_DECIMAIS)
b1 = ax1.bar(medias_gen.index, medias_gen.values, color=sns.color_palette(n_colors=len(medias_gen)))
ax1.bar_label(b1, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax1.axhline(df["Coffee_Intake"].mean(), color="gray", linestyle="--", label="Média geral")
ax1.set_title("H10: Consumo médio por gênero")
ax1.set_xlabel("Gender")
ax1.set_ylabel("Média de Coffee_Intake")
ax1.set_ylim(0, medias_gen.max() * (1 + MARGEM_TOPO_GRAFICO))
ax1.legend(loc="upper right")
ax1.grid(axis="x", visible=False)

medias_oc = df.groupby("Occupation")["Coffee_Intake"].mean().sort_values().round(CASAS_DECIMAIS)
b2 = ax2.barh(medias_oc.index, medias_oc.values, color=sns.color_palette("muted", n_colors=len(medias_oc)))
ax2.bar_label(b2, fmt=f"%.{CASAS_DECIMAIS}f", padding=PADDING_ROTULO)
ax2.axvline(df["Coffee_Intake"].mean(), color="gray", linestyle="--", label="Média geral")
ax2.set_title("H10: Consumo médio por ocupação")
ax2.set_xlabel("Média de Coffee_Intake")
ax2.set_ylabel("Occupation")
ax2.set_xlim(0, medias_oc.max() * (1 + MARGEM_TOPO_GRAFICO))
ax2.legend(loc="lower left", framealpha=0.95)
ax2.grid(axis="y", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

- **Resultado:** As médias de `Coffee_Intake` são praticamente idênticas em todas as dimensões demográficas: entre países variam de **2,41** (`Belgium`) a **2,60** (`Spain`, `F = 0,82`, `p = 0,6894`); por gênero ficam entre **2,49** e **2,52** (`F = 0,37`, `p = 0,6916`); e por ocupação entre **2,48** e **2,54** (`F = 0,41`, `p = 0,8016`). Apenas a taxa de problemas de saúde por país apresentou sinal fraco (**p = 0,0074**, de `35,87%` na França a `46,91%` na Espanha), que deve ser interpretado com cautela por ser um teste isolado entre vários realizados.
- **Conclusão:** **Hipótese rejeitada.** O consumo de café é praticamente uniforme entre países, gêneros e ocupações, e as pequenas variações observadas são compatíveis com flutuação aleatória da amostragem sintética.

## 9. Conclusões